# Behavioral Cloning with CarRacing-v2

Behavioral cloning (BC) is the simplest form of imitation learning: collect expert demonstrations, then train a policy via supervised learning to map observations to actions. It is the starting point for understanding why imitation learning works — and why it fails.

In this notebook you will:

1. Collect expert driving demonstrations
2. Train a BC policy that maps pixel observations to steering/throttle actions
3. Observe **distribution shift** — the core failure mode of BC
4. Fix it with **DAgger** (Dataset Aggregation)

**Environment:** [CarRacing-v2](https://gymnasium.farama.org/environments/box2d/car_racing/) — a continuous-control driving task where the agent observes a 96×96 RGB top-down view of a procedurally generated race track and outputs steering, throttle, and brake.

**Why this environment:**
- Zero setup friction — pip install, runs on CPU, works in Colab
- Visual observations — the policy learns from pixels, not hand-crafted features
- Continuous control — steering and throttle are continuous, matching real driving
- Procedurally generated tracks — each episode is a different track, naturally exposing generalization gaps
- Fast iteration — episodes complete in seconds, enabling rapid experimentation

## Setup

Install the required packages. This cell only needs to run once per Colab session.

In [ ]:
!pip install -q "gymnasium[box2d]" imitation stable-baselines3 swig

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from imitation.data import rollout
from imitation.data.wrappers import RolloutInfoWrapper
from imitation.algorithms import bc

SEED = 42
np.random.seed(SEED)

## Step 1: Create the environment

We wrap the environment in `RolloutInfoWrapper` (needed by the imitation library to track episode boundaries) and `DummyVecEnv` (needed by Stable-Baselines3 for vectorized training).

In [ ]:
def make_env():
    env = gym.make("CarRacing-v2", continuous=True, render_mode="rgb_array")
    env = RolloutInfoWrapper(env)
    return env

venv = DummyVecEnv([make_env])
print(f"Observation space: {venv.observation_space}")
print(f"Action space: {venv.action_space}")

## Step 2: Train an expert policy

We train an expert using PPO. In a real setting you might use a pre-trained checkpoint or human teleoperation. Here PPO acts as our "expert driver."

**Note:** Training for 500k timesteps takes ~15 minutes on a Colab CPU. Reduce `total_timesteps` for a quicker (but weaker) expert.

In [ ]:
expert = PPO(
    "CnnPolicy",
    venv,
    verbose=1,
    seed=SEED,
    n_steps=1024,
    batch_size=64,
    n_epochs=10,
    learning_rate=3e-4,
)

expert.learn(total_timesteps=500_000)
print("Expert training complete")

### Evaluate the expert

Verify that the expert can actually drive before using it to generate demonstrations.

In [ ]:
expert_reward, expert_std = evaluate_policy(expert, venv, n_eval_episodes=10)
print(f"Expert mean reward: {expert_reward:.1f} +/- {expert_std:.1f}")

## Step 3: Collect expert demonstrations

Roll out the expert policy to collect (observation, action) trajectories. These become the training data for behavioral cloning.

In [ ]:
rollouts_data = rollout.rollout(
    expert,
    venv,
    rollout.make_sample_until(min_episodes=50),
    rng=np.random.default_rng(SEED),
)

print(f"Collected {len(rollouts_data)} expert trajectories")
print(f"Total transitions: {sum(len(t) for t in rollouts_data)}")

# Show a sample observation from the expert
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(rollouts_data[0].obs[i * 50])
    ax.set_title(f"Frame {i * 50}")
    ax.axis("off")
fig.suptitle("Sample expert observations")
plt.tight_layout()
plt.show()

## Step 4: Train a behavioral cloning policy

BC treats imitation as supervised learning: minimize the loss between the policy's predicted actions and the expert's recorded actions. The loss function is mean squared error between predicted and expert actions. The policy network is a CNN that processes the 96×96 RGB observation and outputs the three action values (steering, throttle, brake).

In [ ]:
transitions = rollout.flatten_trajectories(rollouts_data)
print(f"Total transitions for BC training: {len(transitions)}")

bc_trainer = bc.BC(
    observation_space=venv.observation_space,
    action_space=venv.action_space,
    demonstrations=transitions,
    rng=np.random.default_rng(SEED),
    batch_size=64,
)

bc_trainer.train(n_epochs=20)
print("BC training complete")

## Step 5: Evaluate and observe distribution shift

Deploy the BC policy and measure its performance against the expert.

**What you will observe:** the BC policy performs noticeably worse than the expert. On straight sections it may track the road adequately, but on sharp turns it drifts off the track. Once off-track, it enters states the expert never demonstrated — predictions become unreliable, and the car spirals further off course.

This is **compounding error** from distribution shift: at training time, the policy only saw states along the expert's trajectory. At test time, any small deviation puts the agent in unfamiliar territory where its predictions are unreliable, causing further deviation.

In [ ]:
bc_reward, bc_std = evaluate_policy(bc_trainer.policy, venv, n_eval_episodes=10)
print(f"Expert mean reward: {expert_reward:.1f} +/- {expert_std:.1f}")
print(f"BC mean reward:     {bc_reward:.1f} +/- {bc_std:.1f}")
print(f"\nPerformance gap:    {expert_reward - bc_reward:.1f} reward points")

### Visualize the performance gap

Compare reward distributions between the expert and BC policies.

In [ ]:
# Collect per-episode rewards for visualization
def collect_episode_rewards(policy, venv, n_episodes=20):
    rewards = []
    for _ in range(n_episodes):
        obs = venv.reset()
        total_reward = 0
        done = False
        while not done:
            action, _ = policy.predict(obs, deterministic=True)
            obs, reward, done, info = venv.step(action)
            total_reward += reward[0]
        rewards.append(total_reward)
    return rewards

expert_rewards = collect_episode_rewards(expert, venv, n_episodes=20)
bc_rewards = collect_episode_rewards(bc_trainer.policy, venv, n_episodes=20)

fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(
    [expert_rewards, bc_rewards],
    labels=["Expert (PPO)", "Behavioral Cloning"],
)
ax.set_ylabel("Episode Reward")
ax.set_title("Expert vs BC: Distribution Shift Degrades Performance")
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

print(f"Expert: {np.mean(expert_rewards):.1f} +/- {np.std(expert_rewards):.1f}")
print(f"BC:     {np.mean(bc_rewards):.1f} +/- {np.std(bc_rewards):.1f}")

## Step 6: Fix it with DAgger

[DAgger](https://arxiv.org/abs/1011.0686) (Dataset Aggregation) addresses distribution shift by iteratively collecting new data from the *learner's* trajectory, labeled by the *expert*. The key idea: let the BC policy drive, but ask the expert what it *would have done* in the states the learner actually visits. Add this data to the training set and retrain.

The algorithm:
1. Train an initial BC policy on expert demonstrations
2. Roll out the learner's policy in the environment
3. Ask the expert to label the states the *learner* actually visited
4. Add this new data to the training set
5. Retrain and repeat

In [ ]:
from imitation.algorithms import dagger
import tempfile

# Re-create a fresh BC trainer for DAgger
bc_trainer_for_dagger = bc.BC(
    observation_space=venv.observation_space,
    action_space=venv.action_space,
    demonstrations=transitions,
    rng=np.random.default_rng(SEED),
    batch_size=64,
)

with tempfile.TemporaryDirectory() as tmpdir:
    dagger_trainer = dagger.SimpleDAggerTrainer(
        venv=venv,
        scratch_dir=tmpdir,
        expert_policy=expert,
        bc_trainer=bc_trainer_for_dagger,
        rng=np.random.default_rng(SEED),
    )

    dagger_trainer.train(total_timesteps=50_000)

print("DAgger training complete")

### Evaluate DAgger

**What you will observe:** the DAgger policy significantly outperforms pure BC, particularly on turns and recovery situations. By training on states the learner actually visits, the policy learns to handle its own mistakes.

In [ ]:
dagger_reward, dagger_std = evaluate_policy(
    dagger_trainer.policy, venv, n_eval_episodes=10
)

print(f"Expert mean reward: {expert_reward:.1f} +/- {expert_std:.1f}")
print(f"BC mean reward:     {bc_reward:.1f} +/- {bc_std:.1f}")
print(f"DAgger mean reward: {dagger_reward:.1f} +/- {dagger_std:.1f}")

### Compare all three policies

In [ ]:
dagger_rewards = collect_episode_rewards(dagger_trainer.policy, venv, n_episodes=20)

fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(
    [expert_rewards, bc_rewards, dagger_rewards],
    labels=["Expert (PPO)", "Behavioral Cloning", "DAgger"],
)
ax.set_ylabel("Episode Reward")
ax.set_title("Expert vs BC vs DAgger")
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

print(f"Expert: {np.mean(expert_rewards):.1f} +/- {np.std(expert_rewards):.1f}")
print(f"BC:     {np.mean(bc_rewards):.1f} +/- {np.std(bc_rewards):.1f}")
print(f"DAgger: {np.mean(dagger_rewards):.1f} +/- {np.std(dagger_rewards):.1f}")

## Summary

| Policy | Training method | Distribution shift? |
|---|---|---|
| **Expert (PPO)** | RL with environment reward | N/A — defines the target distribution |
| **Behavioral Cloning** | Supervised regression on expert data | Yes — compounds errors on unseen states |
| **DAgger** | Iterative BC with learner-visited states labeled by expert | Mitigated — training distribution converges to test distribution |

The distribution shift problem you observed here — and DAgger's fix of relabeling learner-visited states — reappears throughout robot learning. Later in the course, you will revisit these ideas in the context of world models and VLA architectures, where the same fundamental challenge is addressed at larger scale.

## Further reading

- Bagnell (2015). [An Invitation to Imitation](https://www.ri.cmu.edu/pub_files/2015/3/InvitationToImitation_3_1415.pdf) — accessible introduction to the theory
- Ross et al. (2011). [A Reduction of Imitation Learning and Structured Prediction to No-Regret Online Learning](https://arxiv.org/abs/1011.0686) — the DAgger paper
- Bojarski et al. (2016). [End to End Learning for Self-Driving Cars](https://arxiv.org/abs/1604.07316) — NVIDIA's original end-to-end driving work
- The [imitation library documentation](https://imitation.readthedocs.io/en/latest/tutorials/1_train_bc.html) provides complete tutorials for BC, DAgger, GAIL, and AIRL